## Loading preprocessed delta file and creating dim_Employee

In [0]:
# Importing libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id

In [0]:
#Initializing spark session
spark = SparkSession.builder.appName("dimEmployee").getOrCreate()

In [0]:
# File location and type
file_location = "/FileStore/tables/Fact_Sales_Preprocessed"
file_type = "delta"

# The applied options are for delta files. For other file types, these will be ignored.
df_employee = spark.read.format(file_type) \
  .option("inferSchema", True) \
  .option("header", True) \
  .load(file_location)


In [0]:
# Selecting distinct values for "SalesRep", "Department", "EmployeeRole"
df_employee_final = df_employee.select("SalesRep", "Department", "EmployeeRole").distinct()

In [0]:
#Skipping default value and creating surrogate key as we have already handled null values

 #as monotonically_increasing_id() starts with 0, we do monotonically_increasing_id()+1
df_employee_final = df_employee_final.withColumn("Employee_Id", monotonically_increasing_id()+1) \
    .select("Employee_Id","SalesRep", "Department", "EmployeeRole")


In [0]:
df_employee_final.display()

Employee_Id,SalesRep,Department,EmployeeRole
1,Kyle Lin,Electronics,Sales Associate
2,Charles Fields,Apparel,Manager
3,Wendy Castillo,Home,Manager
4,Wendy Castillo,Electronics,Manager
5,Charles Fields,Home,Cashier
6,Kyle Lin,Home,Cashier
7,John Harris,Electronics,Manager
8,John Harris,Apparel,Manager
9,Billy Perez,Electronics,Cashier
10,John Harris,Home,Cashier


In [0]:
df_employee_final.write \
    .format("delta") \
        .mode("overwrite")\
            .save("/FileStore/tables/DIM_Employee")